In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_ollama import OllamaEmbeddings,ChatOllama
from langgraph.types import interrupt,Command
from langgraph.checkpoint.memory import InMemorySaver
from typing import Literal
from pydantic import BaseModel,Field
from langgraph.graph import StateGraph,START,END

In [ ]:
llm=ChatOllama(model="gemma4:31b-cloud")
memory=InMemorySaver()

In [ ]:
class SharedState(BaseModel):
    query:str=''
    draft_email:str=""
    human_feedback:str=""
    final_email:str=''

# **Nodes**

In [ ]:
def draft_Node(state:SharedState)->SharedState:
    if state.human_feedback:
        draft=llm.invoke(f"""
                            Rewrite the email for the query :{state.query},
                            You already wrote this draft email : {state.draft_email},
                            This is the human feedback for the draft email :{state.human_feedback}
                        """)
    else:
        draft=llm.invoke(state.query)
    state.draft_email=draft
    return state

In [ ]:
def human_feedback_Node(state:SharedState)->SharedState:
    feedback=interrupt({
        "draft_email":state.draft_email,
        "question":"if you want to send this email or rewrite the email ?"
    })
    fb=(feedback or "").strip().lower()

    if fb in ("yes","approve","ok","send"):
        state.human_feedback=""
        return state 
    else:
        state.human_feedback=feedback
        return state

In [ ]:
def conditional_routing(state:SharedState)->Literal["draft","final"]:
    if state.human_feedback:
        return "draft"
    else:
        return "final"

In [ ]:
def final_node(state:SharedState)->SharedState:
    if state.human_feedback:
        state.final_email=state.human_feedback
        return state
    else:
        print("Email Sent Successfully...")
        state.final_email="Email Sent Successfully"
        return state

In [ ]:
graph_builder=StateGraph(SharedState)
graph_builder.add_node("draft",draft_Node)
graph_builder.add_node("human",human_feedback_Node)
graph_builder.add_node("final",final_node)

graph_builder.add_edge(START,"draft")
graph_builder.add_edge("draft","human")
graph_builder.add_conditional_edges("human",conditional_routing)
graph_builder.add_edge("final",END)

graph=graph_builder.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())

In [ ]:
config={"configurable":{"thread_id":"1221"}}

In [ ]:
res=graph.invoke({"query":"write an email for medical leave on 4th -9th November"},config=config)
res